# SQL Practice Guide: Two Complete Methods

This guide provides a step-by-step walkthrough to set up and practice SQL on any machine from scratch without permission, server, or credential issues.

---

## Architecture Overview

| Feature | Method 1: Python + Pandas | Method 2: Pure SQL File |
| :--- | :--- | :--- |
| **Primary File** | `sql_pandas_conn.ipynb` | `db_and_table.sql` |
| **Database File** | `diamond.db` | `creation.db` |
| **Engine** | SQLite (`sqlite3` built into Python) | SQLite via VS Code Extension |
| **Execution** | Jupyter Notebook Cells | `Ctrl + Enter` (Database Client) |
| **Output View** | Pandas DataFrame (in Notebook) | Interactive Results Grid (in IDE) |

---

## Method 1: Python + Pandas (`sql_pandas_conn.ipynb`)

Use this method when you want to analyze SQL data using Python, Pandas, and Jupyter Notebooks.

### Step 1: Prerequisites
Ensure `pandas` is installed in your Python environment:
```bash
pip install pandas
```
*(Note: `sqlite3` comes pre-installed with standard Python).*

### Step 2: Code Structure (Split into 3 Cells)

#### Cell 1: Connect and Create Table (Run Once)
```python
import sqlite3
import pandas as pd

# Connects to diamond.db (creates the file automatically if it doesn't exist)
conn = sqlite3.connect("diamond.db", timeout=10)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS diamonds_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    carat REAL,
    cut TEXT,
    color TEXT,
    clarity TEXT,
    price REAL
)
""")
conn.commit()
```

#### Cell 2: Insert Data (Run Once)
```python
cursor.executemany("""
INSERT INTO diamonds_data (carat, cut, color, clarity, price)
VALUES (?, ?, ?, ?, ?)
""", [
    (0.23, 'Ideal', 'E', 'SI2', 326.00),
    (0.21, 'Premium', 'E', 'SI1', 326.00),
    (0.29, 'Premium', 'I', 'VS2', 334.00),
    (0.31, 'Good', 'J', 'SI2', 335.00)
])
conn.commit()
```

#### Cell 3: Practice SQL Queries (Run As Many Times As You Want)
```python
query = "SELECT * FROM diamonds_data WHERE price >= 326.00"
df = pd.read_sql_query(query, conn)
display(df)
```

### Important Tips for Method 1
* **Avoid Duplicates:** Keep Cell 2 separate from Cell 3. If you run `INSERT` every time you display results, rows will continuously duplicate.
* **Avoid "Database is locked" error:** Always use `timeout=10`. If a lock occurs, click the **Restart Kernel (🔄)** button in the notebook toolbar.

---

## Method 2: Pure SQL File (`db_and_table.sql`)

Use this method when you want to write and execute raw SQL queries inside VS Code / Antigravity editor with instant visual table results.

### Step 1: Install the Extension
1. Open the Extensions sidebar (`Ctrl + Shift + X`).
2. Search for **Database Client** (by *cweijan*) and click **Install**.

### Step 2: Initialize `creation.db` (First Time Setup)
> **Crucial Rule:** An SQLite database file must never be a 0-byte blank file (`touch`), or the extension will report "No connection found".

To create a valid database file, run this single line in your terminal:
```bash
python3 -c "import sqlite3; conn = sqlite3.connect('creation.db'); conn.close()"
```

### Step 3: Configure Connection in Database Client
1. Click the **Database icon** (stacked cylinders) on the far-left sidebar.
2. Click the **`+` (Create Connection)** icon at the top of the Database panel.
3. Select **SQLite** from the dropdown list.
4. Fill in:
   * **Database Path:** Click the **Browse (folder)** icon and select `creation.db`, OR paste the absolute path:
     ```text
     /path/to/your/project/mysql/creation.db
     ```
5. Click **Save** / **Connect**.
6. The connection `creation.db` will now appear in your left sidebar.

### Step 4: Write and Execute Queries in `db_and_table.sql`

1. Open `db_and_table.sql`.
2. Write standard SQLite statements:
   ```sql
   -- 1. Create Table
   CREATE TABLE IF NOT EXISTS diamonds_data (
       id INTEGER PRIMARY KEY AUTOINCREMENT,
       carat DECIMAL(4, 2) NOT NULL,
       cut VARCHAR(50),
       color VARCHAR(10),
       clarity VARCHAR(20),
       price DECIMAL(10, 2)
   );

   -- 2. Insert Sample Data
   INSERT INTO diamonds_data (carat, cut, color, clarity, price) VALUES
   (0.23, 'Ideal', 'E', 'SI2', 326.00),
   (0.21, 'Premium', 'E', 'SI1', 326.00),
   (0.29, 'Premium', 'I', 'VS2', 334.00),
   (0.31, 'Good', 'J', 'SI2', 335.00);

   -- 3. Practice Queries
   SELECT * FROM diamonds_data;
   SELECT cut, AVG(price) FROM diamonds_data GROUP BY cut;
   ```

3. **How to Run:**
   * **Run entire file:** Press `Ctrl + A` (select all), then press **`Ctrl + Enter`**.
   * **Run a single query:** Place your cursor on the specific query line and press **`Ctrl + Enter`**.
4. **Where to see results:**
   * A **Result Panel** automatically opens on the right side displaying an interactive table with pagination, sorting, and filtering.

---

## Troubleshooting Cheatsheet

### 1. "No active connection found"
* **Cause:** The active `.sql` file does not know which database to send queries to.
* **Fix:** Look at the bottom-right status bar of VS Code, click **`[Select Database]`**, and pick `creation.db`. Alternatively, right-click `creation.db` in the left panel and click **"New Query"**.

### 2. "Database is locked" (OperationalError)
* **Cause:** An unclosed write transaction is holding the file.
* **Fix:** In Jupyter, restart the kernel (`Kernel -> Restart Kernel`). In Python, always close connections: `conn.close()`.

### 3. Syntax Differences: SQLite vs MySQL
* **Database creation:** In SQLite, do **not** write `CREATE DATABASE` or `USE db;` (the file itself is the database).
* **Auto increment:** In SQLite, write `INTEGER PRIMARY KEY AUTOINCREMENT` instead of `INT AUTO_INCREMENT PRIMARY KEY`.

